In [3]:
!apt-get -qq install fonts-nanum > /dev/null
!pip install -q datasets groq

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

from datasets import load_dataset
import pandas as pd

ds = load_dataset("nvidia/Nemotron-Personas-Korea", split="train")
agents = ds.to_pandas()

POOL_SIZE = 100_000
agents = agents.sample(POOL_SIZE, random_state=42).reset_index(drop=True)

agents_adult = agents[agents['age'] >= 20].reset_index(drop=True)
print(f"전체: {len(agents):,} / 10대 제외: {len(agents_adult):,}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 10.9 MB/s eta 0:00:00


README.md:   0%|          | 0.00/36.0k [00:00<?, ?B/s]

data/train-00000-of-00009.parquet: reconstructing file:   0%|          |  0.00B /  220MB            

data/train-00000-of-00009.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00009.parquet: reconstructing file:   0%|          |  0.00B /  220MB            

...

data/train-00007-of-00009.parquet: downloading bytes:           |  0.00B            

data/train-00008-of-00009.parquet: reconstructing file:   0%|          |  0.00B /  220MB            

data/train-00008-of-00009.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

전체: 100,000 / 10대 제외: 98,894


In [4]:
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('캡스톤'))
print("연결 완료")

연결 완료


In [5]:
import json

TRADE_AREA_INFO = {
  "상권유형": "주거지 인접 골목 근린상권 (동네 생활상권)",
  "대표업종_운영시간": [
    {"업종": "한식·일반음식점", "운영시간": "평일 10시~22시 / 주말 유사 (일부 식당 새벽까지)"},
    {"업종": "슈퍼마켓·마트", "운영시간": "평일·주말 09시~23시"},
    {"업종": "카페", "운영시간": "평일 08시~22시 / 주말 09시~22시"},
    {"업종": "편의점", "운영시간": "24시간 (연중무휴)"},
    {"업종": "숙박시설(모텔·여관)", "운영시간": "24시간"},
    {"업종": "미용실·생활편의", "운영시간": "평일 10시~20시 / 주말 단축·일부 휴무"},
    {"업종": "대형 공원(산책·운동)", "운영시간": "상시 개방 / 평일은 아침·저녁 운동, 주말은 낮 나들이 이용 증가"}
  ],
  "방문목적": ["장보기·생필품 구매", "식사·외식", "일상 커피", "공원 산책·운동·나들이", "거주지 인근 생활", "숙박"],
  "시간특성_이유_평일": "출근 전·퇴근 후 생활 이용이 중심이라 오전과 저녁에 활동이 이어지고, 편의점·숙박·일부 식당의 24시간 운영과 공원 아침 운동으로 새벽·오전 이용도 뒷받침됨 (수치 아님, 업종·장소 특성 기반)",
  "시간특성_이유_주말": "생필품·외식·공원 나들이 수요로 낮 시간대 이용이 평일보다 두터워지는 경향, 심야 유흥 성격은 약함 (수치 아님)",
  "요일특성": "주거지 인접 특성상 요일 편차가 크지 않은 편이나, 공원·외식 수요로 주말 낮 활동이 평일보다 다소 증가하는 경향",
  "한줄요약": "대방역 인근 주거지에 붙은 골목형 근린상권으로, 생활밀착 업종과 대형 공원이 어우러져 평일은 아침·저녁 생활 이용, 주말은 낮 나들이 이용이 두드러지는 동네 상권"
}

AREA_INFO_TEXT = json.dumps(TRADE_AREA_INFO, ensure_ascii=False, indent=2)
print(AREA_INFO_TEXT)
print(f"\n글자수: {len(AREA_INFO_TEXT)}")

{
  "상권유형": "주거지 인접 골목 근린상권 (동네 생활상권)",
  "대표업종_운영시간": [
    {
      "업종": "한식·일반음식점",
      "운영시간": "평일 10시~22시 / 주말 유사 (일부 식당 새벽까지)"
    },
    {
      "업종": "슈퍼마켓·마트",
      "운영시간": "평일·주말 09시~23시"
    },
    {
      "업종": "카페",
      "운영시간": "평일 08시~22시 / 주말 09시~22시"
    },
    {
      "업종": "편의점",
      "운영시간": "24시간 (연중무휴)"
    },
    {
      "업종": "숙박시설(모텔·여관)",
      "운영시간": "24시간"
    },
    {
      "업종": "미용실·생활편의",
      "운영시간": "평일 10시~20시 / 주말 단축·일부 휴무"
    },
    {
      "업종": "대형 공원(산책·운동)",
      "운영시간": "상시 개방 / 평일은 아침·저녁 운동, 주말은 낮 나들이 이용 증가"
    }
  ],
  "방문목적": [
    "장보기·생필품 구매",
    "식사·외식",
    "일상 커피",
    "공원 산책·운동·나들이",
    "거주지 인근 생활",
    "숙박"
  ],
  "시간특성_이유_평일": "출근 전·퇴근 후 생활 이용이 중심이라 오전과 저녁에 활동이 이어지고, 편의점·숙박·일부 식당의 24시간 운영과 공원 아침 운동으로 새벽·오전 이용도 뒷받침됨 (수치 아님, 업종·장소 특성 기반)",
  "시간특성_이유_주말": "생필품·외식·공원 나들이 수요로 낮 시간대 이용이 평일보다 두터워지는 경향, 심야 유흥 성격은 약함 (수치 아님)",
  "요일특성": "주거지 인접 특성상 요일 편차가 크지 않은 편이나, 공원·외식 수요로 주말 낮 활동이 평일보다 다소 증가하는 경향",
  "한줄요약": "대방역 인근 주거지에 붙은 골

In [6]:
def build_prompt(row):
    persona_short = (row.get('persona','')[:150] + row.get('hobbies_and_interests','')[:100])

    return f"""[상권 정보]
{AREA_INFO_TEXT}

[가상 인물]
나이: {row['age']} / 성별: {row['sex']} / 직업: {row['occupation']} / 취미·성향: {persona_short}

위 '상권 정보'와 '인물 특성'을 함께 고려해, 이 인물이 향후 30일간 위 상권을 총 몇 번(0~30 정수) 방문할지,
주로 언제(시간대/평일·주말)인지 판단하세요.
개인 성향(바쁨, 성향, 이 동네와의 접점)을 기준으로 판단하고, 0도 정상적인 답입니다.

시간대는 다음 6개 중 하나만 쓰세요: 00-06, 06-11, 11-14, 14-17, 17-21, 21-24
요일은 평일 또는 주말 중 하나만 쓰세요.

JSON만 출력: {{"방문_횟수": 8, "주요_시간대": "17-21", "주요_요일": "평일"}}"""

sample_prompt = build_prompt(agents_adult.iloc[0])
print(sample_prompt)
print(f"\n프롬프트 전체 글자수: {len(sample_prompt)}")

[상권 정보]
{
  "상권유형": "주거지 인접 골목 근린상권 (동네 생활상권)",
  "대표업종_운영시간": [
    {
      "업종": "한식·일반음식점",
      "운영시간": "평일 10시~22시 / 주말 유사 (일부 식당 새벽까지)"
    },
    {
      "업종": "슈퍼마켓·마트",
      "운영시간": "평일·주말 09시~23시"
    },
    {
      "업종": "카페",
      "운영시간": "평일 08시~22시 / 주말 09시~22시"
    },
    {
      "업종": "편의점",
      "운영시간": "24시간 (연중무휴)"
    },
    {
      "업종": "숙박시설(모텔·여관)",
      "운영시간": "24시간"
    },
    {
      "업종": "미용실·생활편의",
      "운영시간": "평일 10시~20시 / 주말 단축·일부 휴무"
    },
    {
      "업종": "대형 공원(산책·운동)",
      "운영시간": "상시 개방 / 평일은 아침·저녁 운동, 주말은 낮 나들이 이용 증가"
    }
  ],
  "방문목적": [
    "장보기·생필품 구매",
    "식사·외식",
    "일상 커피",
    "공원 산책·운동·나들이",
    "거주지 인근 생활",
    "숙박"
  ],
  "시간특성_이유_평일": "출근 전·퇴근 후 생활 이용이 중심이라 오전과 저녁에 활동이 이어지고, 편의점·숙박·일부 식당의 24시간 운영과 공원 아침 운동으로 새벽·오전 이용도 뒷받침됨 (수치 아님, 업종·장소 특성 기반)",
  "시간특성_이유_주말": "생필품·외식·공원 나들이 수요로 낮 시간대 이용이 평일보다 두터워지는 경향, 심야 유흥 성격은 약함 (수치 아님)",
  "요일특성": "주거지 인접 특성상 요일 편차가 크지 않은 편이나, 공원·외식 수요로 주말 낮 활동이 평일보다 다소 증가하는 경향",
  "한줄요약": "대방역 인근 주

In [7]:
import re, json as jsonlib

def llm_judge_groq(row, max_retries=3):
    prompt = build_prompt(row)
    for _ in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model="openai/gpt-oss-120b",
                messages=[{"role":"user","content":prompt}],
                temperature=0.9,
                max_tokens=700,
            )
            reply = resp.choices[0].message.content.strip()
        except Exception as e:
            print("API 에러:", e)
            continue

        m = re.search(r'\{[^{}]*\}', reply, re.DOTALL)
        if not m:
            continue
        try:
            parsed = jsonlib.loads(m.group(0))
            count = parsed.get('방문_횟수', None)
            time_slot = parsed.get('주요_시간대','').strip()
            weekday = parsed.get('주요_요일','').strip()
            if (isinstance(count,(int,float)) and 0 <= count <= 30
                and time_slot in ['00-06','06-11','11-14','14-17','17-21','21-24']
                and weekday in ['평일','주말']):
                return int(count), time_slot, weekday
        except Exception:
            continue
    return None, None, None

print("함수 준비 완료")

함수 준비 완료


In [8]:
from tqdm import tqdm
import time

SAMPLE_N = 180
sample_final = agents_adult.sample(SAMPLE_N, random_state=11).reset_index(drop=True)

records = []
start = time.time()
for i in tqdm(range(len(sample_final))):
    row = sample_final.iloc[i]
    count, time_slot, weekday = llm_judge_groq(row)
    records.append({'age':row['age'], 'sex':row['sex'], 'count':count, 'time_slot':time_slot, 'weekday':weekday})
    if (i+1) % 50 == 0:
        # 50명마다 중간 저장 (혹시 또 한도 걸려도 데이터 안 날아가게)
        pd.DataFrame(records).to_json(f'partial_{i+1}.json', orient='records', force_ascii=False)

elapsed = time.time() - start
print(f"\n총 처리시간: {elapsed/60:.1f}분")

sample_final['visit_count'] = [r['count'] for r in records]
sample_final['llm_time'] = [r['time_slot'] for r in records]
sample_final['llm_weekday'] = [r['weekday'] for r in records]

fail_count = sample_final['visit_count'].isna().sum()
print(f"판단 실패 건수: {fail_count} / {SAMPLE_N}")

 82%|████████▏ | 148/180 [23:43<03:03,  5.74s/it]

API 에러: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m0w696waen5tkyqjmdm8302q` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 198511, Requested 1763. Please try again in 1m58.368s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
API 에러: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m0w696waen5tkyqjmdm8302q` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 198511, Requested 1763. Please try again in 1m58.368s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
API 에러: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m0w696waen5tkyqjmdm8302q` service tier `on_demand

 83%|████████▎ | 149/180 [23:43<02:06,  4.09s/it]

API 에러: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m0w696waen5tkyqjmdm8302q` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 198511, Requested 1758. Please try again in 1m56.208s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
API 에러: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m0w696waen5tkyqjmdm8302q` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 198511, Requested 1758. Please try again in 1m56.208s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
API 에러: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m0w696waen5tkyqjmdm8302q` service tier `on_demand

 83%|████████▎ | 150/180 [23:43<01:27,  2.93s/it]

...

API 에러: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m0w696waen5tkyqjmdm8302q` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 198495, Requested 1758. Please try again in 1m49.296s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
API 에러: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m0w696waen5tkyqjmdm8302q` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 198495, Requested 1758. Please try again in 1m49.296s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
API 에러: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m0w696waen5tkyqjmdm8302q` service tier `on_demand

 99%|█████████▉| 179/180 [23:50<00:00,  4.26it/s]

API 에러: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m0w696waen5tkyqjmdm8302q` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 198495, Requested 1771. Please try again in 1m54.912s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
API 에러: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m0w696waen5tkyqjmdm8302q` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 198495, Requested 1771. Please try again in 1m54.912s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
API 에러: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m0w696waen5tkyqjmdm8302q` service tier `on_demand

100%|██████████| 180/180 [23:50<00:00,  7.95s/it]

API 에러: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m0w696waen5tkyqjmdm8302q` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 198494, Requested 1758. Please try again in 1m48.864s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
API 에러: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m0w696waen5tkyqjmdm8302q` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 198494, Requested 1758. Please try again in 1m48.864s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
API 에러: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m0w696waen5tkyqjmdm8302q` service tier `on_demand

In [9]:
# 성공한 것만 필터링
sample_final['visit_count'] = [r['count'] for r in records]
sample_final['llm_time'] = [r['time_slot'] for r in records]
sample_final['llm_weekday'] = [r['weekday'] for r in records]

success_only = sample_final[sample_final['visit_count'].notna()].copy()
print(f"성공한 표본: {len(success_only)}명")

# 저장
success_only.to_json('groq_146_final.json', orient='records', force_ascii=False)
from google.colab import files
files.download('groq_146_final.json')

# 분포 미리보기
print(f"\n방문횟수: 최소{success_only['visit_count'].min()} 최대{success_only['visit_count'].max()} 평균{success_only['visit_count'].mean():.1f}")
print(f"0회(비방문): {(success_only['visit_count']==0).sum()}명")

성공한 표본: 146명


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


방문횟수: 최소1.0 최대30.0 평균14.0
0회(비방문): 0명


---
## 3단계: 하이브리드 접근 (개선안)

2단계 결과를 분해해 보면, LLM(120B)은 **방문 횟수**(개인차를 반영한 연속값)에서는 유의미한 판단을 보였으나, **시간대·요일**(범주형 선택)에서는 상권정보 문장을 그대로 재사용하는 경향을 보여 균등 무작위 배정보다도 오차가 컸다. 이에 따라 방문 횟수는 LLM 결과를 유지하고, 시간대·요일은 1단계에서 사용한 규칙 기반(정답표 미참고, 상식적 가정 기반) 확률표로 재배정하는 하이브리드 방식을 구성한다.

**한계 명시**: 아래 확률표(연령대별 시간대 선호, 요일 5:2 비율)는 통계적으로 검증된 자료가 아니라 연구자가 사전에 설정한 상식적 가정이다. 확률표 설계 시점에 실제 정답표(공군호텔 상권의 새벽 유동인구 집중 등 특이 패턴)를 참고하지 않았으나, 이 가정 자체의 일반적 타당성은 별도로 검증되지 않았다는 한계가 있다.

In [ ]:
import numpy as np

# 1단계에서 사용한 연령대별 시간대 확률표 (정답표 미참고, 상식 기반 사전 설계)
TIME_SLOTS = ['00-06','06-11','11-14','14-17','17-21','21-24']
TIME_WEIGHTS = {
    '10대':  [0.02, 0.15, 0.20, 0.25, 0.28, 0.10],
    '20대':  [0.05, 0.10, 0.15, 0.18, 0.32, 0.20],
    '30대':  [0.04, 0.15, 0.18, 0.18, 0.30, 0.15],
    '40대':  [0.04, 0.18, 0.20, 0.20, 0.26, 0.12],
    '50대':  [0.05, 0.22, 0.22, 0.20, 0.21, 0.10],
    '60대+': [0.06, 0.28, 0.24, 0.20, 0.16, 0.06],
}

def age_group(age):
    if age<20: return '10대'
    elif age<30: return '20대'
    elif age<40: return '30대'
    elif age<50: return '40대'
    elif age<60: return '50대'
    else: return '60대+'

success_only['age_group'] = success_only['age'].apply(age_group)

np.random.seed(42)
success_only['hybrid_time'] = success_only['age_group'].apply(
    lambda g: np.random.choice(TIME_SLOTS, p=TIME_WEIGHTS[g])
)

# 요일: 달력 구조(평일 5일:주말 2일) 기반 사전확률
success_only['hybrid_weekday'] = np.random.choice(
    ['평일','주말'], size=len(success_only), p=[5/7, 2/7]
)

print("하이브리드 배정 완료")
print(success_only[['age','visit_count','hybrid_time','hybrid_weekday']].head(10))

## 4. 최종 검증: 하이브리드 결과 vs 실제 정답표 (10대 제외 재정규화)

In [ ]:
def age_group2(age):
    if age<30: return '20대'
    elif age<40: return '30대'
    elif age<50: return '40대'
    elif age<60: return '50대'
    else: return '60대+'
success_only['age_group2'] = success_only['age'].apply(age_group2)

weights = success_only['visit_count']
total_w = weights.sum()

# 성별
sim_gender_raw = success_only.groupby('sex')['visit_count'].sum() / total_w * 100
sim_gender = {'남': sim_gender_raw.get('남자',0), '여': sim_gender_raw.get('여자',0)}

# 연령대
age_order = ['20대','30대','40대','50대','60대+']
sim_age_raw = success_only.groupby('age_group2')['visit_count'].sum() / total_w * 100
sim_age = {g: sim_age_raw.get(g,0) for g in age_order}

# 시간대 (하이브리드)
time_order = ['00-06','06-11','11-14','14-17','17-21','21-24']
sim_time_raw = success_only.groupby('hybrid_time')['visit_count'].sum() / total_w * 100
sim_time = {t: sim_time_raw.get(t,0) for t in time_order}

# 요일 (하이브리드)
sim_wd_raw = success_only.groupby('hybrid_weekday')['visit_count'].sum() / total_w * 100
sim_weekday = {'평일': sim_wd_raw.get('평일',0), '주말': sim_wd_raw.get('주말',0)}

# 정답표 (원본 서울시 상권분석서비스 데이터 기준, 10대 제외 재정규화)
truth_gender = {'남':48.3,'여':51.7}
truth_age_renorm = {'20대':20.23,'30대':20.34,'40대':17.73,'50대':17.53,'60대+':24.17}
truth_time = {'00-06':26.3,'06-11':20.9,'11-14':12.0,'14-17':11.8,'17-21':16.2,'21-24':12.8}
truth_weekday = {'평일':70.97,'주말':29.02}

def compare(name, truth_d, sim_d):
    print(f"\n=== {name} ===")
    errors=[]
    for k in truth_d:
        real, sim = truth_d[k], sim_d.get(k,0)
        err = abs(real-sim); errors.append(err)
        print(f"{k:<8}{real:>8.2f}{sim:>8.2f}{err:>8.2f}")
    mae = sum(errors)/len(errors)
    print(f"MAE: {mae:.2f}%p")
    return mae

mae_g = compare("성별", truth_gender, sim_gender)
mae_a = compare("연령대(10대 제외)", truth_age_renorm, sim_age)
mae_t = compare("시간대(하이브리드)", truth_time, sim_time)
mae_w = compare("요일(하이브리드)", truth_weekday, sim_weekday)

print(f"\n전체 평균 MAE: {(mae_g+mae_a+mae_t+mae_w)/4:.2f}%p")

## 5. 1·2·3단계 종합 비교

| 항목 | 1단계 (순수 규칙기반) | 2단계 (LLM+상권정보) | 3단계 (하이브리드) |
|---|---|---|---|
| 성별 | 1.64%p | 3.61%p | 3.61%p |
| 연령대 (10대 제외) | 2.87%p | 7.71%p | 7.71%p |
| 시간대 | 8.20%p | 18.88%p | 8.02%p |
| 요일 | - | 18.99%p | 0.60%p |
| 전체 평균 | 4.78%p | 12.30%p | 4.99%p |

하이브리드 방식은 요일 오차를 97%, 시간대 오차를 58% 개선하여 1단계 순수 규칙 기반 수준(전체 평균 MAE 4.78%p)에 근접한 4.99%p를 기록했다. 동시에 1단계에는 없었던 개인별 방문 빈도(1~30회) 정보를 유지한다는 점에서, 단순 규칙 기반보다 정보량이 풍부한 모델이다.

**연구의 핵심 기여**는 최저 오차 달성 자체가 아니라, LLM이 연속값 추정(방문 횟수)과 범주형 분류(시간대·요일)에서 서로 다른 신뢰도를 보이며, 특히 공통 배경정보(상권정보)가 개인 정보(페르소나)를 압도해 개인차를 소거시키는 현상을 실증적으로 규명하고, 이를 근거로 한 하이브리드 설계가 실제로 개선 효과가 있음을 검증한 데 있다.